# C3 — Dự báo Hoạt động Đại lý
### Câu hỏi 3: Đại lý nào sẽ mua hàng? | DATA EXPLORERS 2026

**Mục tiêu:**
- Xác suất đại lý đặt hàng trong Q2/2026
- Danh sách đại lý có nguy cơ rời bỏ
- Điểm ưu tiên tiếp thị

> Chạy **C0** trước để sinh `data_raw/fact_sales_full.csv`

# Thư viện và dữ liệu

In [ ]:
# thư viện và dữ liệu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.tsa.stattools import adfuller

# tải dữ liệu giao dịch đầy đủ
df_fact = pd.read_csv('data_raw/fact_sales_full.csv', parse_dates=['order_date'])
df_fact.head()

In [ ]:
# thống kê tổng quan
print(f"Tổng giao dịch : {len(df_fact):,}")
print(f"Đại lý unique  : {df_fact['customer_code'].nunique():,}")
print(f"Tháng          : {df_fact['order_date'].dt.to_period('M').nunique()}")
df_fact.groupby(df_fact['order_date'].dt.to_period('M'))['line_total'].sum()

# Xây dựng đặc trưng RFM

In [ ]:
# ngày tham chiếu = cuối T3/2026
reference_date = pd.Timestamp('2026-03-31')

# Recency — ngày kể từ lần mua cuối
recency = (df_fact.groupby('customer_code')['order_date']
           .max().apply(lambda x: (reference_date - x).days)
           .rename('recency_days'))

In [ ]:
# Frequency — số lần đặt hàng
frequency = (df_fact.groupby('customer_code')['so_number']
             .nunique().rename('frequency'))

In [ ]:
# Monetary — tổng giá trị mua hàng
monetary = (df_fact.groupby('customer_code')['line_total']
            .sum().rename('monetary'))

In [ ]:
# gộp RFM
df_rfm = pd.concat([recency, frequency, monetary], axis=1).reset_index()
df_rfm.head()

In [ ]:
# thêm đặc trưng thời gian
first_order = df_fact.groupby('customer_code')['order_date'].min().rename('first_order')
df_rfm = df_rfm.merge(first_order.reset_index(), on='customer_code')
df_rfm['months_active'] = ((reference_date - df_rfm['first_order']).dt.days / 30).astype(int)

# số đơn 3 tháng gần nhất
last3m = df_fact[df_fact['order_date'] >= '2026-01-01']
last3m_cnt = last3m.groupby('customer_code')['so_number'].nunique().rename('last3m_orders')
df_rfm = df_rfm.merge(last3m_cnt.reset_index(), on='customer_code', how='left')
df_rfm['last3m_orders'] = df_rfm['last3m_orders'].fillna(0)
df_rfm.head()

# Phát hiện tín hiệu rời bỏ

In [ ]:
# nhãn churn: đại lý không đặt hàng trong 60 ngày gần nhất = churn
df_rfm['churn'] = (df_rfm['recency_days'] > 60).astype(int)
print(f"Tỉ lệ churn: {df_rfm['churn'].mean():.1%}")
df_rfm['churn'].value_counts()

In [ ]:
# so sánh hoạt động Q4/2025 vs Q1/2026
q4 = df_fact[df_fact['order_date'].between('2025-10-01','2025-12-31')]
q1 = df_fact[df_fact['order_date'].between('2026-01-01','2026-03-31')]
q4_rev = q4.groupby('customer_code')['line_total'].sum().rename('q4_rev')
q1_rev = q1.groupby('customer_code')['line_total'].sum().rename('q1_rev')
trend = pd.concat([q4_rev, q1_rev], axis=1).fillna(0)
trend['pct_change'] = (trend['q1_rev'] - trend['q4_rev']) / (trend['q4_rev'] + 1) * 100
at_risk = trend[trend['pct_change'] < -30]
print(f"Đại lý giảm >30% từ Q4→Q1: {len(at_risk)}")
at_risk.sort_values('pct_change').head(10)

# Tính dừng (Đại lý lớn nhất)

In [ ]:
# lấy đại lý lớn nhất
top_dealer = df_rfm.nlargest(1,'monetary')['customer_code'].values[0]
ts_dealer  = (df_fact[df_fact['customer_code']==top_dealer]
              .groupby(df_fact['order_date'].dt.to_period('M'))['line_total'].sum())
print(f"Đại lý: {top_dealer} — {len(ts_dealer)} tháng dữ liệu")
ts_dealer

In [ ]:
# kiểm định ADF
pvalue = adfuller(ts_dealer)[1]
if pvalue < 0.05:
    print(f"Chuỗi DỪNG. P-Value = {pvalue:.3f}")
else:
    print(f"Chuỗi KHÔNG DỪNG. P-Value = {pvalue:.3f}")

In [ ]:
# lấy sai phân bậc 1
pvalue = adfuller(ts_dealer.diff().dropna())[1]
if pvalue < 0.05:
    print(f"DỪNG sau sai phân. P-Value = {pvalue:.3f}")
else:
    print(f"KHÔNG DỪNG sau sai phân. P-Value = {pvalue:.3f}")

# Mô hình xác suất mua hàng

In [ ]:
# lấy tham số tốt nhất
parameters = pd.read_csv('Forecasting Product/best_params_classifier.csv', index_col=0)
parameters

In [ ]:
# trích xuất giá trị tham số
n_estimators     = int(parameters.loc['n_estimators'][0])
max_depth        = int(parameters.loc['max_depth'][0])
learning_rate    = float(parameters.loc['learning_rate'][0])
subsample        = float(parameters.loc['subsample'][0])
colsample_bytree = float(parameters.loc['colsample_bytree'][0])

In [ ]:
# xây dựng feature matrix X và nhãn y
feature_cols = ['recency_days','frequency','monetary','months_active','last3m_orders']
X = df_rfm[feature_cols].fillna(0)
y = (df_rfm['recency_days'] <= 90).astype(int)  # mua trong 90 ngày tới

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# tách tập train / test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# mô hình GBM Classifier
model = XGBClassifier(n_estimators=n_estimators,
                      max_depth=max_depth,
                      learning_rate=learning_rate,
                      subsample=subsample,
                      colsample_bytree=colsample_bytree,
                      use_label_encoder=False,
                      eval_metric='logloss',
                      random_state=42)
model.fit(X_train, y_train)

In [ ]:
# kiểm định chéo
cv = StratifiedKFold(n_splits=5, shuffle=False)
cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')
print(f"AUC-ROC trung bình: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Dự báo

In [ ]:
# dự báo xác suất mua hàng Q2/2026
proba = model.predict_proba(X_scaled)[:, 1]
df_result = df_rfm[['customer_code']].copy()
df_result['purchase_prob'] = proba

# phân bố xác suất
df_result['purchase_prob'].hist(bins=20, figsize=(10,4))
plt.title('Phân bố xác suất mua hàng Q2/2026')
plt.xlabel('Xác suất')
plt.show();

# Điểm ưu tiên & Xếp hạng

In [ ]:
# điểm ưu tiên tiếp thị
df_result = df_result.merge(df_rfm[['customer_code','recency_days','monetary','frequency']], on='customer_code')
df_result['priority_score'] = df_result['purchase_prob'] * np.log1p(df_result['monetary'])
df_result['tier'] = pd.cut(df_result['purchase_prob'],
                            bins=[0, 0.3, 0.6, 0.85, 1.0],
                            labels=['RỦI RO', 'TRUNG BÌNH', 'TỐT', 'ƯU TIÊN'])
df_result.sort_values('priority_score', ascending=False).head(10)

In [ ]:
# danh sách đại lý nguy cơ rời bỏ
churn_list = df_result[
    (df_result['purchase_prob'] < 0.3) & (df_result['recency_days'] > 45)
].sort_values('monetary', ascending=False)
print(f"Đại lý nguy cơ rời bỏ: {len(churn_list)}")
churn_list.head(10)

In [ ]:
# Top 20 đại lý ưu tiên tiếp thị
top20 = df_result.nlargest(20, 'priority_score')
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top20['customer_code'], top20['priority_score'], color='steelblue')
ax.set_title('Top 20 Đại lý — Điểm Ưu tiên Tiếp thị Q2/2026')
ax.set_xlabel('Priority Score')
plt.tight_layout()
plt.show();

# Đánh giá

In [ ]:
# báo cáo phân loại
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
# tầm quan trọng đặc trưng
feat_imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
feat_imp.plot(kind='barh', figsize=(8, 4), title='Tầm quan trọng đặc trưng')
plt.tight_layout()
plt.show();

# Xuất kết quả

In [ ]:
# gộp kết quả với thông tin đại lý
df_final = df_result.merge(
    df_fact[['customer_code','customer_name','province_name']].drop_duplicates(),
    on='customer_code', how='left')
df_final.head()

In [ ]:
# xuất kết quả
df_final.to_csv('Forecasting Product/Ensemble/predictions_gbm.csv', index=False)
churn_list.to_csv('data_raw/churn_risk_dealers.csv', index=False)
print('✅ Đã lưu predictions_gbm.csv + churn_risk_dealers.csv')